# IRI-2016 Benchmark (B-01) — Kaggle production workflow

**What this runs.** The project's own benchmark path — `scripts/04_build_external_products.py`
over `src/external/iri.py` — inside the environment the 2026-09-19 Kaggle verification
established (`iricore==1.8.0`, CPython 3.10.12 `virtualenv`, hash-pinned wheels). Steps, each
stopping the notebook on failure after writing the diagnostic bundle:

1. Runtime/platform record; **network preflight** (DNS → TCP → TLS → HTTPS, bounded).
2. Isolated Python 3.10 environment; the four pinned, hash-verified wheels plus `pyyaml`.
   **Step 3b** then installs the project's governed pins (`requirements.txt`) into the SAME
   environment, so the fixture runs and the benchmark share one environment identity
   (D-49 extension, owner-authorized 2026-09-20; `fixture_gate.verify_receipt` is unchanged).
3. Unpack and **hash-verify the project package** (`tec_b01_package.zip`, built by
   `kaggle/build_b01_package.py`; carries `package_manifest.json` with the source commit).
4. **`--verify-runtime`**: the adapter checks the installed release, its default IRI version,
   and the FULL SHA-256 of the two index files it would consume against
   `configs/experiment.yaml: benchmark_b01.index_file_pins` (D-45).
5. **`--build-validation-report`**: the R-59 seven-area report from `b01_validation_samples.json`
   (5–10 samples with official-interface values), using the PREDECLARED tolerance in
   `experiment.yaml`. The adapter's own path is exercised at those points — explicit
   `version=16`, UTC target times, station coordinates, `htop=2000`, TECU.
6. **Fixtures — inside the SAME 3.10 environment** (TE 9.2, TC-03g; D-49 as extended
   2026-09-20): `scripts/run_walking_skeleton.py` runs with the venv's interpreter and the
   package's `--code-commit`. With a **frozen** manifest (`tests/fixtures/<id>/fixture_manifest.yaml`
   + `.sha256`) it is a verification run whose receipt is whatever the orchestrator writes;
   with only an **identity declaration** it is a MEASURING run (`--emit-candidate`), executed
   twice so the runtime/storage RANGE has width (board Rec 5) — the candidate manifest and the
   measuring results are copied into the bundle for the owner's Q-31 freeze act. The
   scientific fixture's measuring run needs a verified plumbing receipt first (R-140), so it
   runs only once the plumbing manifest is frozen. Nothing is marked passed without execution.
7. **`--generate-benchmark`** (gated; **disabled by default**, `RUN_FULL_YEAR = False`): all
   four R-59 limbs over that report, hourly 2022 grid from `data.yaml: stations`, the
   workload with progress, pins re-verified after the session, stamped rows + provenance +
   manifest. Even when enabled it refuses without receipts whose environment identity
   matches this run (D-49 item 4).
8. Bundle everything to `/kaggle/working/b01_bundle.zip`.

**What is real here.** Steps 4–7 call the real installed `iricore` through the real project
adapter (`src/external/iri.py`) via the stage script inside the 3.10 venv. The local checks
recorded in `CR-…-P3` §3.8.3 used a stub `iricore` with the real index bytes: they are
**structural tests** of the gate and output contracts, not runtime evidence; runtime
evidence comes only from this notebook's execution on Kaggle.

**Prerequisites the notebook checks and reports rather than bypasses** (each is an existing
project gate, not a new one): `data.yaml: stations` and `igrf_version` resolved (student
freeze, D-1); `experiment.yaml: benchmark_b01.validation_report.tolerance_tecu` and
`tolerance_declared_at_utc` frozen BEFORE this run; the samples file present with official
values; for step 6, the fixture ladder's own prerequisites (every Phase 1 stage script's refusal
gates — the ladder stops at the first and reports it; the current list is in
`governance/CHANGE_RECORD_2026-09-20_b01_prerequisites.md` §2); for step 7, receipts from
step 6 (same environment identity by construction now) and `RUN_FULL_YEAR = True` set
deliberately in Step 0. Steps 1–5 produce
their evidence regardless.

**Boundaries.** CPU only; no GNSS/VTEC target data is read; nothing under
`evidence/locked_test_restricted/` is packaged or touched; no `write_release`, no
`permitted_producers` registration, no model training. The generated rows are the B-01 product
candidate for the project's release path, not a release.

**Environment note.** The governed interpreter pin is 3.11 (TC-03d); `iricore==1.8.0`'s only
Linux wheel is cp310, so this session runs the benchmark AND its two prerequisite fixture runs
under 3.10.12 in one environment carrying `requirements.txt`'s exact pins plus the hash-pinned
IRI set (D-49 as extended 2026-09-20, owner-authorized; dependency compatibility verified by a
cp310/manylinux dry-run resolution, 50 packages, no conflict). Model training and every other
stage stay on 3.11. The environment lock records the 3.10.12 interpreter and this venv's own
`pip freeze`.


## Step 0 — Helpers (subprocess capture, bundle writer, stop-with-diagnosis) and the run switches

In [ ]:
import datetime as dt
import hashlib
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap
import zipfile
from pathlib import Path

BUNDLE_DIR = Path('/kaggle/working/b01_bundle')
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
VENV_DIR = Path('/kaggle/working/iri_venv')

report = {
    'notebook': 'kaggle_iri2016_benchmark.ipynb',
    'notebook_revision': 'b01-production-3',
    'note': 'B-01 production workflow (revision 2, 2026-09-20): fixtures run inside the same 3.10 environment (D-49 extension); measuring runs when only identity declarations exist',
    'purpose': 'B-01 IRI-2016 benchmark generation through the project adapter, gated by R-59; NOT a release, NOT a registration',
    'generated_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'boundaries': {
        'gnss_vtec_target_data_accessed': False,
        'full_year_benchmark_run': False,
        'producer_artifact_registered': False,
        'g04_passed': False,
    },
}


def run(cmd, *, timeout=900, check=False, env=None):
    """Capture a command's real stdout/stderr/exit code -- never inferred. A timeout is
    itself a recorded outcome (returncode None, timed_out True, partial output kept),
    never an uncaught exception that would lose the report."""
    print('$', ' '.join(cmd) if isinstance(cmd, list) else cmd)
    merged_env = dict(os.environ, **(env or {}))
    entry = {'cmd': cmd if isinstance(cmd, str) else ' '.join(cmd), 'timed_out': False}
    try:
        proc = subprocess.run(
            cmd, shell=isinstance(cmd, str), capture_output=True, text=True,
            timeout=timeout, env=merged_env,
        )
        out, err, rc = proc.stdout or '', proc.stderr or '', proc.returncode
    except subprocess.TimeoutExpired as exc:
        def _s(b):
            return b.decode('utf-8', 'replace') if isinstance(b, bytes) else (b or '')
        out, err, rc = _s(exc.stdout), _s(exc.stderr), None
        entry['timed_out'] = True
        entry['timeout_seconds'] = timeout
        print(f'--- TIMED OUT after {timeout}s (recorded, not raised) ---')
    entry.update({'returncode': rc, 'stdout_tail': out[-4000:], 'stderr_tail': err[-4000:]})
    if rc != 0 or entry['timed_out']:
        # closed-set class (dns_failure / tls_failure / hash_mismatch / ...) so a reader
        # can tell failure kinds apart without the raw log; defined in Step 1b's cell
        entry['failure_class'] = classify_command_failure(err, out, rc, entry['timed_out'])
    print(out[-2000:])
    if err:
        print('--- stderr (tail) ---')
        print(err[-2000:])
    if check and rc != 0:
        raise RuntimeError(f'command failed (exit {rc}): {entry["cmd"]}')
    return entry


def save_and_stop(reason):
    """Write whatever the report holds so far, zip the bundle, then raise -- a clear
    stop with a precise diagnosis, never a silent fallback to something unverified."""
    report['ok'] = False
    report['stopped_reason'] = reason
    write_bundle()
    raise RuntimeError(f'STOPPED: {reason}')


def write_bundle():
    report_path = BUNDLE_DIR / 'verification_report.json'
    report_path.write_text(json.dumps(report, indent=2, default=str), encoding='utf-8')
    zip_path = Path('/kaggle/working/b01_bundle.zip')
    with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
        for p in BUNDLE_DIR.rglob('*'):
            if p.is_file():
                zf.write(p, p.relative_to(BUNDLE_DIR))
    print('Bundle written:', zip_path)
    return zip_path

# --- run switches -----------------------------------------------------------------------
RUN_FULL_YEAR = False   # full-year B-01 generation is DISABLED by default; set True deliberately,
                        # only after the receipts and the validation report exist (owner instruction 2026-09-19)
RUN_FIXTURES = True     # run the fixture ladder: verification runs against frozen manifests, else
                        # MEASURING runs (--emit-candidate) against the packaged identity declarations
report['run_switches'] = {'RUN_FULL_YEAR': RUN_FULL_YEAR, 'RUN_FIXTURES': RUN_FIXTURES}


## Step 1 — Runtime/platform record, then the bounded network preflight

In [ ]:
runtime = {
    'python_version': sys.version,
    'python_version_info': list(sys.version_info),
    'python_executable': sys.executable,
    'platform_platform': platform.platform(),
    'platform_machine': platform.machine(),
    'platform_system': platform.system(),
    'in_kaggle': Path('/kaggle').is_dir(),
}
report['runtime'] = runtime
print(json.dumps(runtime, indent=2))

In [ ]:
"""iri_net_preflight.py -- bounded network preflight and failure classification for the
Kaggle IRI-2016 verification notebook.

Purpose: before anything is installed, establish whether the package index and the
wheel host are reachable, stage by stage (DNS -> TCP -> TLS -> HTTPS), each stage under
its own timeout, and name the first stage that fails. Also classifies the stderr of a
failed install command into a small closed set so a report reader can tell a DNS
failure from a TLS failure from a hash mismatch without reading raw logs.

Inputs: host names and paths; a captured command's stdout/stderr/exit code.
Re-run behaviour: pure network probes and pure string classification; nothing is
written; the caller records the returned dicts.
"""
import socket
import ssl
import time
import urllib.error
import urllib.request

NETWORK_FAILURE_CLASSES = (
    "dns_failure", "tcp_timeout", "tcp_connect_failure", "tls_failure", "tls_timeout",
    "http_error", "http_timeout", "unknown",
)

# Possible causes are stated as possibilities. The preflight cannot see the Kaggle
# settings panel, so it never asserts that the Internet toggle is off.
POSSIBLE_CAUSES = {
    "dns_failure": "name resolution failed: possible causes include the Kaggle notebook's "
                   "Internet setting being OFF (Settings sidebar -> Internet), a DNS outage, "
                   "or a restricted network; this probe cannot tell these apart -- check the "
                   "Internet setting first, then re-run",
    "tcp_timeout": "the name resolved but no TCP connection completed within the timeout: "
                   "possible causes include a firewall, a proxy requirement, or an outage",
    "tcp_connect_failure": "the name resolved but the TCP connection was refused or reset: "
                           "possible causes include a firewall, a proxy requirement, or an outage",
    "tls_failure": "TCP connected but the TLS handshake or certificate validation failed: "
                   "possible causes include a TLS-intercepting proxy, a stale CA bundle, or a "
                   "clock error on the machine",
    "tls_timeout": "TCP connected but the TLS handshake did not complete within the timeout",
    "http_error": "TLS succeeded but the HTTPS request returned an error status: possible "
                  "causes include a blocked path, a proxy error page, or a service incident",
    "http_timeout": "TLS succeeded but the HTTPS response did not arrive within the timeout",
    "unknown": "an unclassified network error; see the recorded exception text",
}


def _elapsed(t0):
    return round(time.monotonic() - t0, 3)


def probe_host(host, port=443, path="/", timeout=10.0, expect_status=None):
    """DNS -> TCP -> TLS -> HTTPS GET, stopping at the first failing stage.

    Returns {"host", "port", "path", "ok", "stages": {...}, "failure_class"?, "error"?}.
    Every stage records its wall time; a failing stage records repr(exception).
    """
    out = {"host": host, "port": port, "path": path, "stages": {}, "ok": False}

    t0 = time.monotonic()
    try:
        infos = socket.getaddrinfo(host, port, type=socket.SOCK_STREAM)
        addrs = sorted({i[4][0] for i in infos})
        out["stages"]["dns"] = {"ok": True, "addresses": addrs[:8], "seconds": _elapsed(t0)}
    except socket.gaierror as exc:
        out["stages"]["dns"] = {"ok": False, "error": repr(exc), "seconds": _elapsed(t0)}
        out["failure_class"] = "dns_failure"
        out["error"] = repr(exc)
        return out

    t0 = time.monotonic()
    try:
        sock = socket.create_connection((host, port), timeout=timeout)
        out["stages"]["tcp"] = {"ok": True, "peer": list(sock.getpeername()[:2]), "seconds": _elapsed(t0)}
    except socket.timeout as exc:
        out["stages"]["tcp"] = {"ok": False, "error": repr(exc), "seconds": _elapsed(t0)}
        out["failure_class"] = "tcp_timeout"
        out["error"] = repr(exc)
        return out
    except OSError as exc:
        out["stages"]["tcp"] = {"ok": False, "error": repr(exc), "seconds": _elapsed(t0)}
        out["failure_class"] = "tcp_connect_failure"
        out["error"] = repr(exc)
        return out

    t0 = time.monotonic()
    try:
        ctx = ssl.create_default_context()
        sock.settimeout(timeout)
        tls = ctx.wrap_socket(sock, server_hostname=host)
        cert = tls.getpeercert() or {}
        subject = dict(x[0] for x in cert.get("subject", ())) if cert.get("subject") else {}
        out["stages"]["tls"] = {
            "ok": True, "version": tls.version(), "cipher": (tls.cipher() or ("",))[0],
            "peer_common_name": subject.get("commonName"), "not_after": cert.get("notAfter"),
            "seconds": _elapsed(t0),
        }
        tls.close()
    except socket.timeout as exc:
        out["stages"]["tls"] = {"ok": False, "error": repr(exc), "seconds": _elapsed(t0)}
        out["failure_class"] = "tls_timeout"
        out["error"] = repr(exc)
        sock.close()
        return out
    except (ssl.SSLError, ssl.CertificateError, OSError) as exc:
        out["stages"]["tls"] = {"ok": False, "error": repr(exc), "seconds": _elapsed(t0)}
        out["failure_class"] = "tls_failure"
        out["error"] = repr(exc)
        sock.close()
        return out

    t0 = time.monotonic()
    url = f"https://{host}{path}"
    req = urllib.request.Request(url, headers={"User-Agent": "iri-verification-preflight/3"})
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            status = resp.status
            resp.read(4096)
        st = {"ok": True, "status": status, "seconds": _elapsed(t0)}
        if expect_status is not None and status != expect_status:
            st.update({"ok": False, "expected_status": expect_status})
            out["stages"]["https"] = st
            out["failure_class"] = "http_error"
            out["error"] = f"HTTP {status} from {url}, expected {expect_status}"
            return out
        out["stages"]["https"] = st
    except urllib.error.HTTPError as exc:
        st = {"ok": False, "status": exc.code, "error": repr(exc), "seconds": _elapsed(t0)}
        if expect_status is None:  # any HTTP answer proves the path end to end
            st["ok"] = True
            out["stages"]["https"] = st
        else:
            out["stages"]["https"] = st
            out["failure_class"] = "http_error"
            out["error"] = repr(exc)
            return out
    except urllib.error.URLError as exc:
        out["stages"]["https"] = {"ok": False, "error": repr(exc), "seconds": _elapsed(t0)}
        reason = exc.reason
        if isinstance(reason, socket.gaierror):
            out["failure_class"] = "dns_failure"
        elif isinstance(reason, (ssl.SSLError, ssl.CertificateError)):
            out["failure_class"] = "tls_failure"
        elif isinstance(reason, socket.timeout) or "timed out" in str(reason):
            out["failure_class"] = "http_timeout"
        else:
            out["failure_class"] = "unknown"
        out["error"] = repr(exc)
        return out
    except socket.timeout as exc:
        out["stages"]["https"] = {"ok": False, "error": repr(exc), "seconds": _elapsed(t0)}
        out["failure_class"] = "http_timeout"
        out["error"] = repr(exc)
        return out

    out["ok"] = True
    return out


def network_preflight(targets=None, timeout=10.0):
    """Probe each target in order; overall ok only if every target is ok.

    Default targets: the PyPI simple index page for iricore (must answer 200) and the
    wheel host (any HTTP answer accepted). Bounded: at most 4 stages x timeout per
    target.
    """
    if targets is None:
        targets = [
            {"host": "pypi.org", "path": "/simple/iricore/", "expect_status": 200},
            {"host": "files.pythonhosted.org", "path": "/", "expect_status": None},
        ]
    results = [probe_host(t["host"], path=t["path"], timeout=timeout, expect_status=t.get("expect_status"))
               for t in targets]
    failed = [r for r in results if not r["ok"]]
    summary = {"ok": not failed, "timeout_seconds_per_stage": timeout, "targets": results}
    if failed:
        first = failed[0]
        summary["failure_class"] = first.get("failure_class", "unknown")
        summary["failed_host"] = first["host"]
        summary["failed_stage"] = next((k for k, v in first["stages"].items() if not v.get("ok")), None)
        summary["error"] = first.get("error")
        summary["possible_causes"] = POSSIBLE_CAUSES.get(summary["failure_class"], POSSIBLE_CAUSES["unknown"])
    return summary


# ---- classifying a failed install command -------------------------------------------

COMMAND_FAILURE_CLASSES = (
    "none", "timeout", "dns_failure", "tls_failure", "connection_failure", "hash_mismatch",
    "platform_tag_mismatch", "resolution_failure", "missing_module", "unknown",
)

_DNS = ("Temporary failure in name resolution", "Name or service not known",
        "nodename nor servname provided", "getaddrinfo failed", "Name resolution failure")
_TLS = ("CERTIFICATE_VERIFY_FAILED", "certificate verify failed", "SSLError", "SSL: ",
        "TLSV1_ALERT", "WRONG_VERSION_NUMBER")
_HASH = ("THESE PACKAGES DO NOT MATCH THE HASHES", "Hashes are required in --require-hashes mode",
         "do not match the hashes")
_PLATFORM = ("is not a supported wheel on this platform", "not supported on this platform")
_CONN = ("Connection refused", "Connection reset", "NewConnectionError", "Max retries exceeded",
         "ProxyError", "Network is unreachable", "ReadTimeoutError", "ConnectTimeoutError")
_RESOLUTION = ("No matching distribution found", "Could not find a version that satisfies")


def classify_command_failure(stderr, stdout="", returncode=None, timed_out=False):
    """Map a failed command's output to one closed-set class. Order matters: a DNS
    failure also prints 'No matching distribution found', so network signatures are
    tested before resolution ones."""
    if timed_out:
        return "timeout"
    if returncode == 0:
        return "none"
    text = (stderr or "") + "\n" + (stdout or "")
    if any(s in text for s in _DNS):
        return "dns_failure"
    if any(s in text for s in _TLS):
        return "tls_failure"
    if any(s in text for s in _HASH):
        return "hash_mismatch"
    if any(s in text for s in _PLATFORM):
        return "platform_tag_mismatch"
    if any(s in text for s in _CONN):
        return "connection_failure"
    if any(s in text for s in _RESOLUTION):
        return "resolution_failure"
    if "No module named" in text:
        return "missing_module"
    return "unknown"


report['network_preflight'] = network_preflight(timeout=10.0)
print(json.dumps(report['network_preflight'], indent=2, default=str))
write_bundle()
if not report['network_preflight']['ok']:
    npf = report['network_preflight']
    save_and_stop(
        f"network preflight failed before any install: host {npf['failed_host']!r}, stage "
        f"{npf['failed_stage']!r}, class {npf['failure_class']!r}: {npf['error']}. "
        f"{npf['possible_causes']}. Nothing was installed; see report['network_preflight'] "
        f"for every stage's outcome and timing."
    )


## Step 2 — Isolated Python 3.10 environment and the pinned wheels (verified 2026-09-19)

In [ ]:
venv_python_source = None
if sys.version_info[:2] == (3, 10):
    venv_python_source = sys.executable
    strategy = 'kernel_python_is_3.10_isolate_via_venv'
else:
    which = shutil.which('python3.10')
    if which:
        venv_python_source = which
        strategy = 'found_existing_python3.10_isolate_via_venv'
    else:
        # No system python3.10: Step 3 will obtain a complete CPython 3.10 through uv's
        # managed interpreters (an immutable published python-build-standalone release),
        # and only as a last resort through apt-get.
        strategy = 'no_system_python3.10_use_uv_managed_3.10'

report['environment_strategy'] = {
    'kernel_python_minor': sys.version_info[:2],
    'strategy': strategy,
    'venv_python_source': venv_python_source,
}
print(json.dumps(report['environment_strategy'], indent=2, default=str))

write_bundle()  # partial report on disk before anything is installed

In [ ]:
env_logs = report.setdefault('installation', {}).setdefault('isolated_env_creation', [])
venv_python = str(VENV_DIR / 'bin' / 'python')

def env_ready():
    return Path(venv_python).is_file() and run([venv_python, '-m', 'pip', '--version'])['returncode'] == 0

mechanism = None

def fresh():
    if VENV_DIR.exists():
        shutil.rmtree(VENV_DIR)

def attempt(label, cmd, **kw):
    env_logs.append({'attempt': label, **run(cmd, **kw)})
    write_bundle()  # the diagnosis is on disk after EVERY attempt

# Rung 1: virtualenv against the image's own python3.10 (only if one exists).
if venv_python_source is not None:
    fresh()
    attempt('rung1: pip install virtualenv into kernel python',
            [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', 'virtualenv'], timeout=300)
    attempt('rung1: probe the image python3.10 itself',
            [venv_python_source, '-c', 'import sys, sysconfig; print(sys.version); print(sysconfig.get_paths()["stdlib"])'], timeout=60)
    attempt('rung1: virtualenv -p python3.10',
            [sys.executable, '-m', 'virtualenv', '-p', venv_python_source, str(VENV_DIR)], timeout=300)
    if env_ready():
        mechanism = 'virtualenv against the image python3.10'

# Rung 2: a uv-managed CPython 3.10 (complete standalone interpreter, immutable release).
if mechanism is None:
    fresh()
    attempt('rung2: pip install uv into kernel python',
            [sys.executable, '-m', 'pip', 'install', '--quiet', '--no-input', 'uv==0.12.17'], timeout=300)
    uv_env = {'UV_PYTHON_INSTALL_DIR': '/kaggle/working/uv_python', 'UV_CACHE_DIR': '/kaggle/working/uv_cache'}
    attempt('rung2: uv python install cpython-3.10.21',
            [sys.executable, '-m', 'uv', 'python', 'install', 'cpython-3.10.21'], timeout=600, env=uv_env)
    attempt('rung2: uv python list (what uv will use)',
            [sys.executable, '-m', 'uv', 'python', 'list', '--only-installed'], timeout=120, env=uv_env)
    attempt('rung2: uv venv --seed --python 3.10',
            [sys.executable, '-m', 'uv', 'venv', '--seed', '--python', 'cpython-3.10.21', str(VENV_DIR)], timeout=300, env=uv_env)
    if env_ready():
        mechanism = 'uv 0.12.17 managed cpython-3.10.21 (python-build-standalone release) + uv venv --seed'

# Rung 3 (last resort): Debian's python3.10-venv, non-interactive, bounded.
if mechanism is None and venv_python_source is not None:
    fresh()
    attempt('rung3: apt-get install python3.10-venv (noninteractive)',
            ['bash', '-lc', 'export DEBIAN_FRONTEND=noninteractive; apt-get update -qq && apt-get install -y -qq --no-install-recommends python3.10-venv python3.10-distutils'],
            timeout=240)
    attempt('rung3: python3.10 -m venv', [venv_python_source, '-m', 'venv', str(VENV_DIR)], timeout=300)
    if env_ready():
        mechanism = 'stdlib venv after apt-get python3.10-venv'

if mechanism is None:
    summary = chr(10).join(
        f"  - {e['attempt']}: exit={e['returncode']} timed_out={e.get('timed_out')} "
        f"class={e.get('failure_class', 'none')} "
        f"stderr_tail={(e.get('stderr_tail') or e.get('stdout_tail') or '')[-400:].strip()!r}"
        for e in env_logs
    )
    save_and_stop(
        'no isolated Python 3.10 environment could be created: every rung failed (virtualenv against the image python3.10 / uv-managed CPython 3.10 / apt-get + stdlib venv) -- see report["installation"]["isolated_env_creation"] for each attempt: exact command, exit code, stdout and stderr. Per-attempt summary:' + chr(10) + summary
    )

venv_info = run([venv_python, '-c', 'import sys, platform; print(sys.version); print(platform.platform())'])
report['environment_strategy']['venv_python_version'] = venv_info['stdout_tail'].strip()
report['environment_strategy']['isolated_env_mechanism'] = mechanism
write_bundle()
print(json.dumps(report['environment_strategy'], indent=2, default=str))

In [ ]:
# Exact published wheel hashes (fetched from PyPI's JSON API before this run; the
# only Linux/cp310 wheel each package published at the pinned version).
REQUIREMENTS_TXT = textwrap.dedent('''\
    numpy==1.26.4 --hash=sha256:ffa75af20b44f8dba823498024771d5ac50620e6915abac414251bd971b4529f
    fortranformat==2.0.3 --hash=sha256:88c8e7a3eac16c23420e8a1c4b21ddc7108f48e8dcbd2e0da6c8ecc48b051bb2
    pymap3d==3.2.0 --hash=sha256:fccd44f2f6021a95adec19771c603b8dac104eab120d863c463d76b9bc298669
    iricore==1.8.0 --hash=sha256:f452b22316891d87ee766dba266de6a07e4e6008ab515ffed902ea8b5446a874
    pyyaml==6.0.1 --hash=sha256:ba336e390cd8e4d1739f42dfe9bb83a3cc2e80f567d8805e11b46f4a943f5515
''')
req_path = BUNDLE_DIR / 'requirements-iri.txt'
req_path.write_text(REQUIREMENTS_TXT, encoding='utf-8')
print(REQUIREMENTS_TXT)

pip_log = run(
    [venv_python, '-m', 'pip', 'install', '--no-deps', '--require-hashes', '-r', str(req_path)],
    timeout=600,
)
report.setdefault('installation', {})['pip_install'] = pip_log
if pip_log['returncode'] != 0:
    save_and_stop(
        f'pip install --require-hashes failed inside the isolated venv: failure_class='
        f'{pip_log.get("failure_class")!r} (dns_failure / tls_failure / connection_failure = network; '
        f'hash_mismatch = downloaded bytes differ from the pinned SHA-256; platform_tag_mismatch = '
        f'the manylinux_2_35 wheel is refused by this glibc; resolution_failure = no matching file). '
        f'See report["installation"]["pip_install"] for the captured stdout/stderr and exit code -- '
        f'reported exactly, never silently retried with a different version.'
    )

## Step 3 — Unpack and hash-verify the project package

`tec_b01_package.zip` is looked for under `/kaggle/input/**` (attach it as a Kaggle dataset)
or `/kaggle/working/`. Kaggle **auto-extracts** a `.zip` uploaded as a dataset, so the package is
equally accepted as its already-unpacked tree — the directory under `/kaggle/input/**` that holds
`package_manifest.json`. Either way, every file is checked against the package's own manifest
before use; for the extracted form there are no archive bytes to hash, so `package.sha256` is
`null` and the per-file manifest check plus `tree_sha256` are the integrity evidence.

In [ ]:
import glob

WORKSPACE = Path('/kaggle/working/tec_workspace')
# Kaggle auto-extracts a .zip uploaded as a dataset, so the package arrives either as the zip
# itself or as its already-unpacked tree (the directory holding package_manifest.json).
zips = sorted(glob.glob('/kaggle/input/**/tec_b01_package.zip', recursive=True)) + \
       sorted(glob.glob('/kaggle/working/tec_b01_package.zip'))
trees = sorted(glob.glob('/kaggle/input/**/package_manifest.json', recursive=True))
if not zips and not trees:
    save_and_stop('tec_b01_package.zip (or its extracted tree holding package_manifest.json) not found under '
                  '/kaggle/input/** or /kaggle/working/; build it with python kaggle/build_b01_package.py, '
                  'upload the zip as a Kaggle dataset and attach that dataset to this notebook (Add Input)')
if WORKSPACE.exists():
    shutil.rmtree(WORKSPACE)
if zips:
    pkg = Path(zips[0])
    pkg_form = 'zip'
    pkg_sha = hashlib.sha256(pkg.read_bytes()).hexdigest()
    WORKSPACE.mkdir(parents=True)
    with zipfile.ZipFile(pkg) as zf:
        zf.extractall(WORKSPACE)
else:
    pkg = Path(trees[0]).parent
    pkg_form = 'extracted'
    pkg_sha = None  # no archive bytes to hash; the per-file manifest check below is the integrity evidence
    shutil.copytree(pkg, WORKSPACE)
manifest = json.loads((WORKSPACE / 'package_manifest.json').read_text(encoding='utf-8'))
bad = {}
for rel, expected in manifest['files'].items():
    actual = hashlib.sha256((WORKSPACE / rel).read_bytes()).hexdigest()
    if actual != expected:
        bad[rel] = {'expected': expected, 'actual': actual}
report['package'] = {'path': str(pkg), 'form': pkg_form, 'sha256': pkg_sha, 'source_commit': manifest['source_commit'], 'tree_sha256': manifest.get('tree_sha256'),
                     'built_at_utc': manifest['built_at_utc'], 'file_count': len(manifest['files']),
                     'hash_mismatches': bad}
write_bundle()
if bad:
    save_and_stop(f'package files do not match package_manifest.json: {bad}')
print('package verified:', json.dumps({k: v for k, v in report['package'].items() if k != 'hash_mismatches'}, indent=2))
SCRIPT = WORKSPACE / 'scripts' / '04_build_external_products.py'
CONFIGS = WORKSPACE / 'configs'
B01_OUT = WORKSPACE / 'artifacts' / 'external' / 'b01'
STAGE_ENV = {'TEC_WORKSPACE_ROOT': str(WORKSPACE), 'PYTHONHASHSEED': '0', 'PYTHONPATH': str(WORKSPACE)}
CODE_COMMIT = manifest['source_commit']


def stage04(*flags, timeout):
    """Run the stage script inside the venv; capture, record, and copy its outputs into the bundle."""
    entry = run([venv_python, str(SCRIPT), '--config', str(CONFIGS), '--code-commit', CODE_COMMIT, *flags],
                timeout=timeout, env=STAGE_ENV)
    report.setdefault('stage04', []).append({'flags': list(flags), **entry})
    if B01_OUT.exists():
        for f in B01_OUT.iterdir():
            if f.is_file():
                shutil.copyfile(f, BUNDLE_DIR / f.name)
    write_bundle()
    return entry


## Step 3b — The governed pins into the SAME environment (D-49 extension, 2026-09-20)

`requirements.txt` (numpy, pandas, pyyaml, scikit-learn, tensorflow, pytest, ruff — exact pins)
is installed into the 3.10 venv that already carries the hash-pinned IRI set. Every pin has a
cp310 manylinux wheel and `iricore`'s own constraints (`numpy<2`, `fortranformat`, `pymap3d`)
are satisfied by the same pins — verified by a dry-run resolution before this revision was
built (50 packages, no conflict). The `pip freeze` recorded below is the one the environment
lock of every stage run in this session carries, so fixture receipts and the benchmark run
share one environment identity by construction (`fixture_gate.verify_receipt` unchanged).


In [ ]:
req = WORKSPACE / 'requirements.txt'
if not req.is_file():
    save_and_stop('requirements.txt is not in the package')
e = run([venv_python, '-m', 'pip', 'install', '--no-input', '-r', str(req)], timeout=3600)
report['installation']['pip_install_requirements'] = e
write_bundle()
if e['returncode'] != 0:
    save_and_stop(f"pip install -r requirements.txt failed inside the venv (class {e.get('failure_class')}): "
                  f"{(e['stderr_tail'] or '')[-1500:]}")
# the IRI set must still be at its pins after the governed pins landed (same numpy pin; nothing upgraded)
chk = run([venv_python, '-c', "import importlib.metadata as m; print({p: m.version(p) for p in ('iricore','numpy','fortranformat','pymap3d','pyyaml','tensorflow','pandas','scikit-learn','pytest')})"])
report['installation']['post_requirements_versions'] = chk['stdout_tail'].strip()
freeze = run([venv_python, '-m', 'pip', 'freeze'])
report['installation']['pip_freeze'] = freeze['stdout_tail']   # recorded AFTER every install
write_bundle()
print(chk['stdout_tail'])


## Step 4 — Runtime pin protection (`--verify-runtime`)

The adapter verifies the installed release, its default IRI version and the **full** SHA-256 of
the `apf107.dat` / `ig_rz.dat` it would consume against `experiment.yaml: benchmark_b01`.
A mismatch stops here: the run never continues past a failed pin.


In [ ]:
e = stage04('--verify-runtime', timeout=600)
if e['returncode'] != 0:
    save_and_stop(f"--verify-runtime refused (exit {e['returncode']}, class {e.get('failure_class')}): "
                  f"{(e['stderr_tail'] or '')[-1500:]}")
identity = json.loads((B01_OUT / 'b01_runtime_identity.json').read_text(encoding='utf-8'))
report['runtime_identity'] = identity
print(json.dumps({k: identity[k] for k in ('release', 'installed_default_iri_version', 'index_files_sha256', 'python')}, indent=2))


## Step 5 — The R-59 validation report (`--build-validation-report`)

Needs `b01_validation_samples.json` in the package (5–10 samples with official IRI-2016
interface values; template packaged) and the predeclared tolerance frozen in `experiment.yaml`.
The adapter path is exercised at every sample (explicit `version=16`, UTC time, station
coordinate, `htop=2000`, TECU). A failed report is written and stops the run — never hidden.


In [ ]:
samples = WORKSPACE / 'kaggle' / 'b01_validation_samples.json'
report_path = B01_OUT / 'iri_implementation_validation_report.json'
if not samples.is_file():
    # revision 2: NOT a stop -- the fixture ladder (Step 6) is independent of the official
    # reference values and may run in the same session; Step 7 stays gated on the report.
    report['validation'] = {'status': 'not run', 'reason': 'kaggle/b01_validation_samples.json absent from the package: '
                            'fill kaggle/b01_validation_samples.TEMPLATE.json with the official IRI-2016 interface values '
                            '(R-59 area 6), freeze the tolerance in experiment.yaml (area 7), rebuild the package'}
    write_bundle()
    print('validation report NOT RUN:', report['validation']['reason'])
else:
    e = stage04('--build-validation-report', str(samples), timeout=1800)
    if report_path.is_file():
        report['validation'] = json.loads(report_path.read_text(encoding='utf-8'))
    if e['returncode'] != 0:
        save_and_stop(f"validation report did not pass (exit {e['returncode']}): {(e['stderr_tail'] or '')[-1500:]}")
    print('validation report:', report['validation']['status'], '| samples:',
          [(s['site'], s['abs_diff'], s['within_tolerance']) for s in report['validation']['samples']])


## Step 6 — Walking-skeleton fixtures in THIS environment (TE 9.2, TC-03g; D-49 extension)

Runs `scripts/run_walking_skeleton.py` with the venv's interpreter and the package's
`--code-commit`. Per fixture: a **frozen** manifest → verification run (receipt written only by
the orchestrator on a pass); an **identity declaration** only → two MEASURING runs
(`--emit-candidate --identity …`; the first persists its measuring result and refuses to
compose a zero-width range, the second composes the candidate — board Rec 5). The scientific
fixture requires a verified plumbing receipt (R-140), so its measuring run happens only in a
session where the plumbing manifest is frozen. The ladder stops at the first stage refusal and
the exact stderr is recorded — that refusal is the deliverable of an incomplete ladder, never a
bypass. Candidate manifests, measuring results and receipts are copied into the bundle.


In [ ]:
FIXTURE_IDS = ('plumbing_7day', 'scientific_1month')
FIX_ROOT = WORKSPACE / 'tests' / 'fixtures'
state = {}
for fid in FIXTURE_IDS:
    state[fid] = {
        'frozen_manifest': (FIX_ROOT / fid / 'fixture_manifest.yaml').is_file() and (FIX_ROOT / fid / 'fixture_manifest.sha256').is_file(),
        'candidate_manifest': (FIX_ROOT / fid / 'fixture_manifest.yaml').is_file() and not (FIX_ROOT / fid / 'fixture_manifest.sha256').is_file(),
        'identity_declaration': (FIX_ROOT / fid / 'identity_declaration.yaml').is_file(),
    }
report['fixtures'] = {'state_in_package': state, 'runs': []}
skel_env = {'TEC_WORKSPACE_ROOT': str(WORKSPACE), 'PYTHONHASHSEED': '0', 'PYTHONPATH': str(WORKSPACE), 'CUDA_VISIBLE_DEVICES': ''}
SKEL = WORKSPACE / 'scripts' / 'run_walking_skeleton.py'


def copy_fixture_outputs(fid):
    out = WORKSPACE / 'artifacts' / 'walking_skeleton' / fid
    dest = BUNDLE_DIR / 'fixtures' / fid
    dest.mkdir(parents=True, exist_ok=True)
    if out.exists():
        for f in out.rglob('*'):
            if f.is_file():
                target = dest / f.relative_to(out)
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(f, target)
    for name in ('fixture_manifest.yaml', 'fixture_manifest.sha256'):
        p = FIX_ROOT / fid / name
        if p.is_file():
            shutil.copyfile(p, dest / name)


def skeleton(fid, *extra, label):
    e = run([venv_python, str(SKEL), '--config', str(CONFIGS), '--fixture', fid, '--code-commit', CODE_COMMIT,
             '--python', venv_python, *extra], timeout=4 * 3600, env=skel_env)
    report['fixtures']['runs'].append({'fixture': fid, 'step': label, **e})
    copy_fixture_outputs(fid)
    write_bundle()
    return e


if not RUN_FIXTURES:
    report['fixtures']['status'] = 'not run (RUN_FIXTURES is False)'
else:
    status = []
    for fid in FIXTURE_IDS:
        st = state[fid]
        if st['frozen_manifest']:
            e = skeleton(fid, label='verification run against the frozen manifest')
            status.append(f"{fid}: verification run exit {e['returncode']}")
            if e['returncode'] != 0:
                status.append(f"{fid}: did not pass; the receipt, if any, is whatever the orchestrator wrote -- ladder stops here")
                break
        elif st['identity_declaration'] and not st['candidate_manifest']:
            if fid == 'scientific_1month' and not state['plumbing_7day']['frozen_manifest']:
                status.append('scientific_1month: measuring run NOT attempted -- it requires a verified plumbing receipt (R-140), i.e. a FROZEN plumbing manifest; freeze plumbing first')
                break
            decl = FIX_ROOT / fid / 'identity_declaration.yaml'
            e1 = skeleton(fid, '--emit-candidate', '--identity', str(decl), label='measuring run 1 (expected: measuring result persisted; zero-width range refused)')
            if 'zero-width' not in (e1['stderr_tail'] or '') and e1['returncode'] != 0:
                status.append(f"{fid}: measuring run 1 stopped at a stage refusal (exit {e1['returncode']}); see stderr_tail -- ladder stops here")
                break
            e2 = skeleton(fid, '--emit-candidate', '--identity', str(decl), label='measuring run 2 (expected: candidate manifest composed)')
            status.append(f"{fid}: measuring runs exit {e1['returncode']}/{e2['returncode']}; candidate={ (FIX_ROOT / fid / 'fixture_manifest.yaml').is_file() }")
            if e2['returncode'] != 0:
                break
            # a candidate is not a receipt: the owner's Q-31 freeze act (status frozen + sha256 sidecar
            # + D-number) happens outside this notebook; the ladder cannot continue past it here
            status.append(f"{fid}: candidate written -- owner freeze act required before a receipt can exist")
            break
        else:
            status.append(f"{fid}: no frozen manifest and no identity declaration in the package (or a candidate without its freeze) -- not run")
            break
    report['fixtures']['status'] = '; '.join(status)
write_bundle()
print('fixtures:', report['fixtures']['status'])


## Step 7 — Gated generation (`--generate-benchmark`) — disabled by default

`RUN_FULL_YEAR = False` unless set deliberately in Step 0. When enabled, the stage script still
refuses without receipts whose recorded environment identity matches this run's lock — which,
since the fixtures now run in this same environment (D-49 extension), is satisfiable by
construction once both manifests are frozen and both verification runs pass in this session.
The full year is 3 × 8,760 = 26,280 calls (TC-04).


In [ ]:
receipts = sorted(glob.glob(str(WORKSPACE / 'artifacts' / 'walking_skeleton' / '*' / 'fixture_pass_receipt.json')))
report['receipts_present'] = receipts
if not RUN_FULL_YEAR:
    report['generation'] = {'status': 'disabled', 'reason': 'RUN_FULL_YEAR is False (default): this run performs the runtime, fixture and reference-validation checks only'}
elif len(receipts) < 2:
    report['generation'] = {'status': 'skipped', 'reason': 'TE 9.2 fixture receipts absent from this workspace; the stage script would refuse'}
elif not report_path.is_file():
    report['generation'] = {'status': 'skipped', 'reason': 'no R-59 validation report was produced in this session (Step 5); the stage script would refuse'}
else:
    e = stage04('--generate-benchmark', '--validation-report', str(report_path), timeout=6 * 3600)
    prov_path = B01_OUT / 'b01_provenance.json'
    if prov_path.is_file():
        report['generation'] = json.loads(prov_path.read_text(encoding='utf-8'))
    if e['returncode'] != 0:
        save_and_stop(f"--generate-benchmark refused (exit {e['returncode']}): {(e['stderr_tail'] or '')[-1500:]}")
write_bundle(); print('generation:', report['generation'].get('status', 'ran'))


## Step 8 — Bundle

Download `/kaggle/working/b01_bundle.zip`: `verification_report.json` (this notebook's record),
`b01_runtime_identity.json`, `iri_implementation_validation_report.json`, the fixture receipts when step 6 ran, and — when step 7 ran —
`b01_iri2016_rows.jsonl`, `b01_provenance.json`, `sha256_manifest.json`, plus the experiment
registry rows the stage script appended under `tec_workspace/artifacts/registry/`.


In [ ]:
reg = WORKSPACE / 'artifacts' / 'registry'
if reg.exists():
    for f in reg.iterdir():
        if f.is_file():
            shutil.copyfile(f, BUNDLE_DIR / ('registry_' + f.name))
report['ok'] = True
zip_path = write_bundle()
print('DONE. Download:', zip_path)
